# 08 — Inferencia en Datos Reales

Aplicamos los modelos entrenados al dataset real del invernadero.

**Importante:** No hay métricas RMSE/MAE válidas aquí — no tenemos ground truth. La validación es cualitativa, por inspección visual de un experto de dominio.

El objetivo es identificar y caracterizar anomalías **reales** en el período 2024-03-06 → 2025-03-07.

**Entrada:** CSV real + modelos de `data/models/`  
**Salida:** gráficas en `data/plots/`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar modelos entrenados

In [ ]:
rf_detector          = joblib.load(MODELO_1_PATH)
imputer              = joblib.load(IMPUTER_M1_PATH)
features_esperadas   = joblib.load(FEATURES_M1_PATH)
rf_clasificador_tipo = joblib.load(MODELO_2_PATH)
label_encoder_model2 = joblib.load(LABEL_ENC_M2_PATH)
iterative_imputer_faltantes = joblib.load(IMPUTER_FALT_PATH)
print("Todos los modelos cargados.")


## 1. Cargar y preparar el dataset real
Cargamos el mismo dataset combinado filtrado al año de análisis.

In [ ]:
# 1. Definir la ruta a la carpeta donde están los CSV
ruta_carpeta = 'data/combined_*.csv'
archivos_csv = glob.glob(ruta_carpeta)

if not archivos_csv:
    print("No se encontraron archivos CSV. Verifica que la carpeta 'Completados' exista en el mismo directorio que este notebook.")
else:
    columnas_por_archivo = {}
    
    # 2. Leer los encabezados de cada archivo
    for archivo in archivos_csv:
        nombre_archivo = os.path.basename(archivo)
        try:
            # Usamos nrows=0 para leer solo los encabezados y no saturar la memoria
            df = pd.read_csv(archivo, nrows=0) 
            columnas_por_archivo[nombre_archivo] = set(df.columns)
        except Exception as e:
            print(f"Error al leer {nombre_archivo}: {e}")

    # 3. Mostrar las columnas de cada archivo
    print("=== COLUMNAS POR ARCHIVO ===")
    for archivo, columnas in columnas_por_archivo.items():
        print(f"\n📁 {archivo} ({len(columnas)} columnas):")
        print(list(columnas))

## 2. Preparar features sobre datos reales
Aplicamos las mismas transformaciones que durante el entrenamiento.

In [ ]:
# Ingeniería de características

# Diccionario donde guardaremos cada base de datos procesada por separado.
# La llave será el nombre del archivo y el valor será el DataFrame listo.
datasets_listos = {}

if not archivos_csv:
    print("No se encontraron archivos CSV.")
else:
    for archivo in archivos_csv:
        nombre_dataset = os.path.basename(archivo).replace('.csv', '')
        
        try:
            print(f"\n=======================================================")
            print(f"Procesando base de datos: {nombre_dataset}")
            print(f"=======================================================")
            
            # 1. Cargar el DataFrame individual
            df_temp = pd.read_csv(archivo)
            
            # 2. Manejo de Fechas
            df_temp['Fecha'] = pd.to_datetime(df_temp['Fecha'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
            fechas_nulas = df_temp['Fecha'].isnull().sum()
            if fechas_nulas > 0:
                print(f"  ⚠️ Advertencia: {fechas_nulas} fechas no se pudieron leer. Revisa el formato.")
            else:
                print(f"  📅 Fechas leídas correctamente (Formato Día/Mes/Año)")

            # 3. Extraer características temporales
            # Filtrar al mismo período que el entrenamiento (1 año completo)
            df_temp = df_temp[df_temp['Fecha'] <= '2025-03-07'].copy()
            print(f"  📅 Filtrado hasta 2025-03-07: {len(df_temp)} filas")
            
            df_temp['Hora'] = df_temp['Fecha'].dt.hour
            df_temp['DiaSemana'] = df_temp['Fecha'].dt.dayofweek
            df_temp['Mes'] = df_temp['Fecha'].dt.month
            
            # 4. Preparar las columnas para las predicciones
            df_temp['etiqueta_deteccion'] = 'normal' 
            df_temp['etiqueta_tipo_anomalia'] = pd.NA 
            
            # 5. Guardar en el diccionario
            datasets_listos[nombre_dataset] = df_temp
            
            print(f"  ✅ Procesado exitosamente. Filas: {len(df_temp)}")
            
        except Exception as e:
            print(f"  ❌ Error al procesar {nombre_dataset}: {e}")

print("\n=== RESUMEN DE BASES DE DATOS LISTAS PARA EVALUAR ===")
for nombre in datasets_listos.keys():
    print(f"- {nombre}")

## 3. Cargar modelo 1 y detectar anomalías reales

In [ ]:
# Cargamos el imputer y el modelo 1 (ya cargados en celda 0)
try:
    imputer_cargado     = joblib.load(IMPUTER_M1_PATH)
    rf_detector_cargado = joblib.load(MODELO_1_PATH)
    features_esperadas  = joblib.load(FEATURES_M1_PATH)
    print("Modelos cargados correctamente.")
except FileNotFoundError as e:
    print(f"Error cargando modelos: {e}")


## 4. Visualizar detecciones del Modelo 1 por sensor

In [ ]:
# 1. Definir la lista de todas las variables que quieres revisar
variables_a_graficar = ['XTS', 'PCO2EXT', 'XHINV', 'PHEXT', 'PRGINT', 'PRAD', 
                        'PTEXT', 'XCO2I', 'XTINV', 'UVENT_cen', 'UVENT_lN', 'PVV']

# 2. Elegir el dataset
nombre_evaluar = 'combined_2024_03_06-2025_11_30_1min'
df_visual = datasets_listos[nombre_evaluar].copy()

# 3. Bucle para generar todas las gráficas
for variable in variables_a_graficar:
    # Verificamos si la variable existe en este dataset para evitar errores
    if variable in df_visual.columns:
        plt.figure(figsize=(15, 5))
        
        # Filtramos normales y anomalías
        normales = df_visual[df_visual['etiqueta_deteccion'] == 'normal']
        anomalias = df_visual[df_visual['etiqueta_deteccion'] == 'anomalia']
        
        # Dibujamos puntos normales
        plt.scatter(normales['Fecha'], normales[variable], 
                    color='blue', label='Normal', alpha=0.4, s=10)
        
        # Dibujamos anomalías en rojo
        plt.scatter(anomalias['Fecha'], anomalias[variable], 
                    color='red', label='Anomalía Detectada', alpha=0.8, s=20)
        
        # Configuración de la gráfica
        plt.title(f"Detección de Anomalías - Dataset: {nombre_evaluar}\nVariable: {variable}")
        plt.xlabel("Fecha")
        plt.ylabel(f"Valor de {variable}")
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True, linestyle='--', alpha=0.5) # Añadimos rejilla para mejor lectura
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ La variable {variable} no se encuentra en el dataset.")

## 5. Zoom en período concreto para revisión manual

In [ ]:
# 1. Definir la variable específica que quieres revisar
variable = 'XHINV'

# 2. Elegir el dataset
nombre_evaluar = 'combined_2024_03_06-2025_11_30_1min'
df_visual = datasets_listos[nombre_evaluar].copy()

# 3. Asegurar que la columna 'Fecha' sea un objeto datetime para poder filtrar por rango
df_visual['Fecha'] = pd.to_datetime(df_visual['Fecha'], dayfirst=True, errors='coerce')

# 4. Filtrar por el rango de fechas solicitado 
fecha_inicio = '2024-10-28 18:30:00'
fecha_fin = '2024-10-29 06:00:00'

df_filtrado = df_visual[(df_visual['Fecha'] >= fecha_inicio) & (df_visual['Fecha'] <= fecha_fin)]

# 5. Generar la gráfica
if variable in df_filtrado.columns:
    plt.figure(figsize=(15, 5))
    
    # Filtramos normales y anomalías usando el DataFrame YA FILTRADO por fechas
    normales = df_filtrado[df_filtrado['etiqueta_deteccion'] == 'normal']
    anomalias = df_filtrado[df_filtrado['etiqueta_deteccion'] == 'anomalia']
    
    # Dibujamos puntos normales
    plt.scatter(normales['Fecha'], normales[variable], 
                color='blue', label='Normal', alpha=0.4, s=10)
    
    # Dibujamos anomalías en rojo
    plt.scatter(anomalias['Fecha'], anomalias[variable], 
                color='red', label='Outlier', alpha=0.8, s=20)
    
    # Configuración de la gráfica
    plt.xlabel("Date") # Cambiado para reflejar fecha y hora
    plt.ylabel(variable) # Ajustado a la variable dinámicamente
    plt.legend()
    
    # <-- NUEVA CONFIGURACIÓN PARA FECHA Y HORA -->
    # Forzamos a que el eje X muestre Día/Mes/Año Hora:Minuto
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%Y %H:%M'))
    
    plt.xticks
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ La variable {variable} no se encuentra en el dataset.")

## 6. Cargar Modelo 2 y clasificar tipos

In [ ]:
print("=== INICIANDO CLASIFICACIÓN DE TIPOS DE ANOMALÍAS (MODELO 2) ===")

# Cargar el Modelo 2 y el Label Encoder usando las rutas de config
try:
    rf_clasificador_cargado = joblib.load(MODELO_2_PATH)
    label_encoder_cargado   = joblib.load(LABEL_ENC_M2_PATH)
    print("✅ Modelo 2 y Label Encoder cargados correctamente.\n")
except FileNotFoundError as e:
    print(f"❌ Error: No se encontró un archivo del Modelo 2: {e}")
    raise

# Iterar sobre las bases de datos ya evaluadas por el Modelo 1
for nombre, df in {k: v for k, v in datasets_listos.items() if k.startswith('combined_')}.items():
    print(f"\nClasificando anomalías en: {nombre}")

    mascara_anomalias = df['etiqueta_deteccion'] == 'anomalia'
    df_solo_anomalias = df[mascara_anomalias]

    if df_solo_anomalias.empty:
        print("  -> No se detectaron anomalías en este dataset. Se omite el Modelo 2.")
        continue

    try:
        X_anomalias = df_solo_anomalias[features_esperadas].select_dtypes(include='number').copy()
        X_anom_imputed_np = imputer_cargado.transform(X_anomalias)

        predicciones_numericas = rf_clasificador_cargado.predict(X_anom_imputed_np)
        predicciones_texto = label_encoder_cargado.inverse_transform(predicciones_numericas)

        df.loc[mascara_anomalias, 'etiqueta_tipo_anomalia'] = predicciones_texto

        conteo_tipos = df.loc[mascara_anomalias, 'etiqueta_tipo_anomalia'].value_counts()
        print("  -> Tipos de anomalías encontradas:")
        for tipo, cantidad in conteo_tipos.items():
            print(f"     - {tipo}: {cantidad} registros")

    except Exception as e:
        print(f"  ❌ Error al clasificar en {nombre}: {e}")

print("\n✅ Proceso de Clasificación (Modelo 2) Finalizado.")


## 7. Visualizar clasificaciones del Modelo 2 por sensor

In [ ]:
# 1. Definir las variables a revisar
variables_a_graficar = ['XTS', 'PCO2EXT', 'XHINV', 'PHEXT', 'PRGINT', 'PRAD', 
                        'PTEXT', 'XCO2I', 'XTINV', 'UVENT_cen', 'UVENT_lN', 'PVV']

# 2. Elegir el dataset que ya pasó por el Modelo 2
nombre_evaluar = 'combined_2024_03_06-2025_11_30_1min'
df_visual = datasets_listos[nombre_evaluar].copy()

# 3. Bucle para generar las gráficas con leyenda multiclase
for variable in variables_a_graficar:
    if variable in df_visual.columns:
        plt.figure(figsize=(15, 6))
        
        # --- A. GRAFICAR PUNTOS NORMALES ---
        # Usamos un color neutro (gris o azul claro) para que no distraigan
        normales = df_visual[df_visual['etiqueta_deteccion'] == 'normal']
        plt.scatter(normales['Fecha'], normales[variable], 
                    color='lightgrey', label='Normal', alpha=0.3, s=10)
        
        # --- B. GRAFICAR ANOMALÍAS POR TIPO ---
        # Obtenemos solo las filas que son anomalías
        df_anomalias = df_visual[df_visual['etiqueta_deteccion'] == 'anomalia']
        
        # Identificamos qué tipos de anomalías encontró el Modelo 2
        tipos_encontrados = df_anomalias['etiqueta_tipo_anomalia'].unique()
        
        # Usamos un mapa de colores para asignar uno distinto a cada tipo
        cmap = plt.get_cmap('tab10') 
        
        for i, tipo in enumerate(tipos_encontrados):
            # Filtramos el subset de este tipo específico de anomalía
            subset = df_anomalias[df_anomalias['etiqueta_tipo_anomalia'] == tipo]
            
            plt.scatter(subset['Fecha'], subset[variable], 
                        color=cmap(i), 
                        label=f"Anomalía: {tipo}", 
                        edgecolors='black', # Borde negro para que resalten
                        linewidths=0.5,
                        s=30)
        
        # --- CONFIGURACIÓN DE LA GRÁFICA ---
        plt.title(f"Clasificación de Anomalías (Modelo 2) - {nombre_evaluar}\nVariable: {variable}")
        plt.xlabel("Fecha")
        plt.ylabel(f"Valor de {variable}")
        
        # Colocamos la leyenda fuera de la gráfica si hay muchos tipos
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), title="Categorías")
        
        plt.xticks(rotation=45)
        plt.grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ Variable {variable} no encontrada.")

## 8. Corrección — Datos Faltantes

> **Nota:** RMSE/MAE aquí no mide calidad real — solo cuánto cambia el valor. No hay ground truth.

In [ ]:

# NOTA: En Fase 7 (datos reales sin inyección) no existe ground truth conocido.
# El RMSE/MAE aquí NO mide calidad de corrección — mide cuánto cambia el valor detectado.
# La evaluación cuantitativa válida está en Fases 1-6 (datos sintéticos con ground truth).
# Aquí la validación es cualitativa: inspección visual por experto del dominio.
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'DATOS FALTANTES' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Sincronización Crítica) ---
nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'

if nombre_objetivo not in datasets_listos:
    raise KeyError(f"Dataset {nombre_objetivo} no disponible.")

df_trabajo = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

# Función de limpieza que ya probamos que funciona para sincronizar
def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para que df_trabajo y df_original coincidan perfectamente
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_trabajo = preparar_y_redondear(df_trabajo)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. IDENTIFICACIÓN DE COLUMNAS ---
columnas_sensores = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
                     'XCO2I', 'XHINV', 'XTINV', 'XTS', 'UVENT_cen', 'UVENT_lN']
cols_imputar = [col for col in columnas_sensores if col in df_trabajo.columns]

# --- 3. CONFIGURACIÓN DEL IMPUTADOR ---
# Usamos RandomForest para capturar relaciones no lineales entre sensores
iterative_imputer = IterativeImputer(
    estimator=RandomForestRegressor(**IMPUTER_RF_PARAMS),
    max_iter=IMPUTER_MAX_ITER,
    random_state=42,
    initial_strategy='median'
)

# --- 4. EJECUCIÓN DE LA IMPUTACIÓN ---
print(f"Ejecutando IterativeImputer sobre: {cols_imputar}")
# Entrenamos y transformamos
df_imputado_np = iterative_imputer.fit_transform(df_trabajo[cols_imputar])
df_referencia_imputada = pd.DataFrame(df_imputado_np, columns=cols_imputar, index=df_trabajo.index)

# Aplicamos la corrección solo donde: 1. Es NaN  Y  2. El modelo dijo "Datos Faltantes"
mask_pred_faltantes = df_trabajo['etiqueta_tipo_anomalia'] == 'Datos Faltantes'

all_true, all_imputed = [], []

for col in cols_imputar:
    mask_nan = df_trabajo[col].isnull()
    mask_final = mask_pred_faltantes & mask_nan
    
    if mask_final.any():
        idx_a_corregir = df_trabajo.index[mask_final]
        
        # --- EVALUACIÓN (Sincronizada) ---
        idx_comun = idx_a_corregir.intersection(df_original_eval.index)
        if not idx_comun.empty:
            t_vals = df_original_eval.loc[idx_comun, col]
            i_vals = df_referencia_imputada.loc[idx_comun, col]
            m_valid = t_vals.notna() & i_vals.notna()
            all_true.extend(t_vals[m_valid].tolist())
            all_imputed.extend(i_vals[m_valid].tolist())
        
        # Aplicar el valor imputado al dataset real
        df_trabajo.loc[idx_a_corregir, col] = df_referencia_imputada.loc[idx_a_corregir, col]

# --- 5. MÉTRICAS Y GRÁFICA ---
if all_true:
    rmse = np.sqrt(mean_squared_error(all_true, all_imputed))
    print(f"\n✅ Imputación evaluada sobre {len(all_true)} puntos.")
    print(f"RMSE Global de Imputación: {rmse:.3f}")

    plt.figure(figsize=(7, 7))
    plt.scatter(all_true, all_imputed, alpha=0.4, s=10, c='blue', label='Valores Imputados')
    lims = [min(all_true), max(all_true)]
    plt.plot(lims, lims, 'r--', label='Ideal')
    plt.title("Calidad de Imputación: Real vs IterativeImputer")
    plt.xlabel("Valor Real (Original)"); plt.ylabel("Valor Imputado")
    plt.legend(); plt.grid(True); plt.show()
else:
    print("\n⚠️ No se encontraron puntos coincidentes para evaluar (revisa los rangos de fecha).")

# Guardar progreso
datasets_listos[nombre_objetivo] = df_trabajo.reset_index()
print("\n✅ FASE 1 COMPLETADA.")

## 9. Corrección — Sensor Atascado

In [ ]:

# NOTA: En Fase 7 (datos reales sin inyección) no existe ground truth conocido.
# El RMSE/MAE aquí NO mide calidad de corrección — mide cuánto cambia el valor detectado.
# La evaluación cuantitativa válida está en Fases 1-6 (datos sintéticos con ground truth).
# Aquí la validación es cualitativa: inspección visual por experto del dominio.
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'SENSOR ATASCADO' ===")

# --- 1. FUNCIONES DE CORRECCIÓN (Mantenidas de tu código original) ---

def marcar_segmentos_stuck_como_nan_condicional(df_a_marcar, columna_nombre, predicciones_series, min_duracion_atasco=5):
    valores = df_a_marcar[columna_nombre].values
    indices_df = df_a_marcar.index
    n = len(valores)
    segmentos_nanificados = []
    nan_count = 0
    i = 0
    while i < n:
        if pd.isna(valores[i]):
            i += 1
            continue
        j = i
        while j < n and pd.notna(valores[j]) and valores[j] == valores[i]:
            j += 1
        duracion = j - i
        
        if duracion >= min_duracion_atasco:
            indices_segmento = indices_df[i:j]
            # Verificamos si en este segmento el modelo predijo "Sensor Atascado"
            if (predicciones_series.loc[indices_segmento] == "Sensor Atascado").any():
                df_a_marcar.loc[indices_segmento, columna_nombre] = np.nan
                segmentos_nanificados.append((indices_segmento[0], indices_segmento[-1]))
                nan_count += 1
                i = j
                continue
        i += 1
    return nan_count, segmentos_nanificados

def corregir_dinamicos_con_estacional_simple(df_a_corregir, df_historico, columna_a_corregir, start_idx, end_idx, ventana_dias=7):
    # Extraemos el segmento dañado
    segmento_indices = df_a_corregir.loc[start_idx:end_idx].index
    
    for idx_anomalo in segmento_indices:
        hora_anomala = idx_anomalo.hour
        fecha_anomala = idx_anomalo.date()
        
        # Buscamos en los días anteriores
        min_date = fecha_anomala - pd.Timedelta(days=ventana_dias + 5)
        max_date = fecha_anomala - pd.Timedelta(days=1)
        
        # Filtramos el histórico para esa hora específica
        mask_historico = (df_historico.index.date >= min_date) & (df_historico.index.date <= max_date) & (df_historico.index.hour == hora_anomala)
        valores_historicos = df_historico.loc[mask_historico, columna_a_corregir].dropna()
        
        if len(valores_historicos) >= 1:
            df_a_corregir.loc[idx_anomalo, columna_a_corregir] = valores_historicos.median()

# --- 2. PREPARACIÓN DE DATOS ---
nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'
df_hibrido = datasets_listos[nombre_objetivo].copy()

# Es vital tener el índice como Datetime para la corrección estacional e interpolación
if 'Fecha' in df_hibrido.columns and not isinstance(df_hibrido.index, pd.DatetimeIndex):
    df_hibrido['Fecha'] = pd.to_datetime(df_hibrido['Fecha'], errors='coerce')
    df_hibrido.set_index('Fecha', inplace=True)
    df_hibrido = df_hibrido.sort_index()

df_original_eval = df_original.copy()
if 'Fecha' in df_original_eval.columns and not isinstance(df_original_eval.index, pd.DatetimeIndex):
    df_original_eval['Fecha'] = pd.to_datetime(df_original_eval['Fecha'], errors='coerce')
    df_original_eval.set_index('Fecha', inplace=True)
    df_original_eval = df_original_eval.sort_index()

serie_predicciones = df_hibrido['etiqueta_tipo_anomalia']

# --- 3. SEPARACIÓN DE VARIABLES ---
columnas_dinamicas = [col for col in ['PRAD', 'PRGINT', 'UVENT_cen', 'UVENT_lN', 'PVV'] if col in df_hibrido.columns]
columnas_estables = [col for col in ['XTS', 'PCO2EXT', 'XHINV', 'PHEXT', 'PTEXT', 'XCO2I', 'XTINV'] if col in df_hibrido.columns]

print(f"-> Columnas Dinámicas (Estacional): {columnas_dinamicas}")
print(f"-> Columnas Estables (Interpolación): {columnas_estables}")

segmentos_dinamicos_corregidos = []
segmentos_estables_corregidos = []

# --- 4. CORRECCIÓN VARIABLES DINÁMICAS (Método Estacional) ---
print("\n--- Corrigiendo Atascos en Columnas Dinámicas ---")
for col in columnas_dinamicas:
    num_marcados, lista_segmentos = marcar_segmentos_stuck_como_nan_condicional(df_hibrido, col, serie_predicciones)
    if num_marcados > 0:
        print(f"  -> {col}: {num_marcados} segmentos atascados detectados.")
        for start_idx, end_idx in lista_segmentos:
            corregir_dinamicos_con_estacional_simple(df_hibrido, df_original_eval, col, start_idx, end_idx)
            segmentos_dinamicos_corregidos.append((col, start_idx, end_idx))

# --- 5. CORRECCIÓN VARIABLES ESTABLES (Interpolación) ---
print("\n--- Corrigiendo Atascos en Columnas Estables ---")
for col in columnas_estables:
    num_marcados, lista_segmentos = marcar_segmentos_stuck_como_nan_condicional(df_hibrido, col, serie_predicciones)
    if num_marcados > 0:
        print(f"  -> {col}: {num_marcados} segmentos atascados detectados e interpolados.")
        df_hibrido[col] = df_hibrido[col].interpolate(method='linear', limit_direction='both')
        for start_idx, end_idx in lista_segmentos:
            segmentos_estables_corregidos.append((col, start_idx, end_idx))

# --- 6. IMPUTACIÓN FINAL (Limpieza de NaNs residuales) ---
print("\n--- Ejecutando SimpleImputer Final para NaNs residuales ---")

columnas_objetivo = columnas_dinamicas + columnas_estables

# Seleccionamos las columnas numéricas que se van a imputar
cols_imputer = [col for col in columnas_objetivo if col in df_hibrido.columns]
nans_antes_final = df_hibrido[cols_imputer].isnull().sum().sum()

if nans_antes_final > 0:
    print(f"  -> Detectados {nans_antes_final} NaNs residuales (no resueltos por interpolación o estacionalidad).")
    print("  -> Entrenando y aplicando Imputador Iterativo localmente...")
    
    # Creamos un imputador fresco y lo entrenamos (fit_transform) directamente con estos datos
    imputador_final = IterativeImputer(
        estimator=RandomForestRegressor(**IMPUTER_RF_PARAMS),
        max_iter=IMPUTER_MAX_ITER,
        random_state=42,
        initial_strategy='median'
    )
    
    imputed_array = imputador_final.fit_transform(df_hibrido[cols_imputer])
    df_imputed_temp = pd.DataFrame(imputed_array, index=df_hibrido.index, columns=cols_imputer)
    
    for col in cols_imputer:
        df_hibrido[col] = df_hibrido[col].fillna(df_imputed_temp[col])
        
    print(f"  -> Imputación final completada. NaNs restantes: {df_hibrido[cols_imputer].isnull().sum().sum()}")
else:
    print("  -> No hay NaNs residuales. Omitiendo paso.")

# --- 7. EVALUACIÓN DE CORRECCIONES (Versión Corregida) ---
def evaluar_correcciones(segmentos, nombre_metodo):
    if not segmentos: 
        print(f" No se detectaron segmentos para {nombre_metodo}. No hay nada que graficar.")
        return
    
    all_true = []
    all_imputed = []
    
    # Usamos una copia para no alterar el original durante la comparación
    df_ref = df_original_eval
    
    for col, start_idx, end_idx in segmentos:
        try:
            # Extraemos los rangos
            true_vals = df_ref.loc[start_idx:end_idx, col]
            imputed_vals = df_hibrido.loc[start_idx:end_idx, col]
            
            # Alineación crítica por índice
            common_idx = true_vals.index.intersection(imputed_vals.index)
            if common_idx.empty: continue

            t_v = true_vals.loc[common_idx]
            i_v = imputed_vals.loc[common_idx]
            
            # Solo comparamos donde ambos tengan valores numéricos (evitar NaNs en el original)
            mask = t_v.notna() & i_v.notna()
            
            all_true.extend(t_v[mask].values)
            all_imputed.extend(i_v[mask].values)
        except Exception as e:
            print(f" Error evaluando segmento {col} ({start_idx}): {e}")

    # --- AQUÍ ESTÁ EL CAMBIO DE LÓGICA ---
    if len(all_true) > 0:
        # 1. Calculamos métricas primero
        rmse = np.sqrt(mean_squared_error(all_true, all_imputed))
        mae = mean_absolute_error(all_true, all_imputed)
        
        print(f"\nResultados Globales - {nombre_metodo}:")
        print(f"  -> RMSE: {rmse:.3f} | MAE: {mae:.3f} (sobre {len(all_true)} puntos)")

        # 2. Generamos la gráfica (DENTRO del bloque if len > 0)
        plt.figure(figsize=(6, 6))
        plt.scatter(all_true, all_imputed, alpha=0.5, s=15, 
                    c='green' if 'Interpolación' in nombre_metodo else 'orange')
        
        min_val = min(min(all_true), min(all_imputed))
        max_val = max(max(all_true), max(all_imputed))
        
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal')
        plt.title(f"Reales vs Corregidos ({nombre_metodo})")
        plt.xlabel("Valores Reales")
        plt.ylabel("Valores Corregidos")
        plt.legend()
        plt.grid(True)
        plt.show() # Ahora sí mostrará la gráfica con datos
    else:
        print(f"⚠️ No se encontraron puntos coincidentes o válidos para comparar en {nombre_metodo}.")

# Llamada a las funciones
evaluar_correcciones(segmentos_dinamicos_corregidos, "Método Estacional (Dinámicos)")
evaluar_correcciones(segmentos_estables_corregidos, "Interpolación (Estables)")

# Guardar y resetear índice
df_hibrido = df_hibrido.reset_index() 
datasets_listos[nombre_objetivo] = df_hibrido

## 10. Corrección — Ruido

In [ ]:

# NOTA: En Fase 7 (datos reales sin inyección) no existe ground truth conocido.
# El RMSE/MAE aquí NO mide calidad de corrección — mide cuánto cambia el valor detectado.
# La evaluación cuantitativa válida está en Fases 1-6 (datos sintéticos con ground truth).
# Aquí la validación es cualitativa: inspección visual por experto del dominio.
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'RUIDO' (ANOMALÍAS PUNTUALES) ===")

# --- 1. PREPARACIÓN CON REDONDEO (Indispensable para que haya gráficas) ---
nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'
df_ruido = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeamos al minuto para asegurar que df_ruido y df_original coincidan exactamente
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_ruido = preparar_y_redondear(df_ruido)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. LÓGICA DE CORRECCIÓN ---
mask_ruido = df_ruido['etiqueta_tipo_anomalia'] == "Ruido"
all_true_ruido, all_imputed_ruido = [], []
puntos_corregidos_resumen = {}

if not mask_ruido.any():
    print(f"\nNo se detectaron anomalías de tipo 'Ruido'.")
else:
    # Iteramos solo por las columnas que existen en el dataset
    cols_a_procesar = [c for c in columnas_objetivo if c in df_ruido.columns]
    
    for col in cols_a_procesar:
        # Calculamos el promedio de los vecinos (anterior y posterior)
        val_prev = df_ruido[col].shift(1)
        val_next = df_ruido[col].shift(-1)
        promedio_vecinos = (val_prev + val_next) / 2
        
        # Identificamos los puntos a corregir en esta columna
        mask_a_corregir = mask_ruido & df_ruido[col].notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_ruido.index[mask_a_corregir]
            
            # --- EVALUACIÓN CONTRA ORIGINAL ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            if not idx_comun.empty:
                t_vals = df_original_eval.loc[idx_comun, col]
                i_vals = promedio_vecinos.loc[idx_comun]
                # Solo agregamos si ninguno es NaN para no romper el RMSE
                mask_valid = t_vals.notna() & i_vals.notna()
                all_true_ruido.extend(t_vals[mask_valid].tolist())
                all_imputed_ruido.extend(i_vals[mask_valid].tolist())
            
            # --- APLICAR LA CORRECCIÓN AL DATASET ---
            df_ruido.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_corregidos_resumen[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y GRÁFICA FINAL ---
    print("\nResumen de Correcciones:")
    for col, count in puntos_corregidos_resumen.items():
        print(f"   - {col}: {count} ruidos corregidos.")

    if all_true_ruido:
        rmse_ruido = np.sqrt(mean_squared_error(all_true_ruido, all_imputed_ruido))
        mae_ruido = mean_absolute_error(all_true_ruido, all_imputed_ruido)
        print(f"\n✅ Evaluación exitosa sobre {len(all_true_ruido)} puntos.")
        print(f"RMSE Global Ruido: {rmse_ruido:.3f} | MAE: {mae_ruido:.3f}")

        plt.figure(figsize=(6, 6))
        plt.scatter(all_true_ruido, all_imputed_ruido, alpha=0.6, color='red', s=30, label='Valores Corregidos')
        
        # Línea de identidad (Ideal)
        mn = min(min(all_true_ruido), min(all_imputed_ruido))
        mx = max(max(all_true_ruido), max(all_imputed_ruido))
        plt.plot([mn, mx], [mn, mx], 'k--', lw=2, label="Ideal")
        
        plt.title("Evaluación Modelo 3: Corrección de Ruido")
        plt.xlabel("Valor Real (Dataset Original)")
        plt.ylabel("Valor Corregido (Promedio Vecinos)")
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        plt.show()

# Guardar resultados y resetear el índice para mantener el formato original
df_ruido = df_ruido.reset_index()
datasets_listos[nombre_objetivo] = df_ruido

print("\n✅ FASE 3 COMPLETADA. El dataset ha sido purificado de Ruido.")

## 11. Corrección — Fuera de Rango

In [ ]:

# NOTA: En Fase 7 (datos reales sin inyección) no existe ground truth conocido.
# El RMSE/MAE aquí NO mide calidad de corrección — mide cuánto cambia el valor detectado.
# La evaluación cuantitativa válida está en Fases 1-6 (datos sintéticos con ground truth).
# Aquí la validación es cualitativa: inspección visual por experto del dominio.
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'DESVIACIÓN DE CORRELACIÓN' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Indispensable para sincronizar) ---
nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'
df_corr_dev = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para asegurar que df_corr_dev y df_original coincidan
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_corr_dev = preparar_y_redondear(df_corr_dev)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. LÓGICA DE CORRECCIÓN ---
factor_umbral_diferencia_corr_dev = FACTOR_UMBRAL_CORRECCION
nombre_clase_corr_dev = "Desviación de Correlación"

# Máscara global según la predicción del modelo
mask_corr_dev = df_corr_dev['etiqueta_tipo_anomalia'] == nombre_clase_corr_dev

puntos_corregidos_por_columna = {}
all_true_corr_dev = []
all_imputed_corr_dev = []

if not mask_corr_dev.any():
    print(f"\nNo se detectaron anomalías de tipo '{nombre_clase_corr_dev}'.")
else:
    print(f"-> Filas marcadas como Desviación: {mask_corr_dev.sum()}")
    
    for col in [c for c in columnas_objetivo if c in df_corr_dev.columns]:
        std_dev_col = df_original_eval[col].std()
        umbral = factor_umbral_diferencia_corr_dev * std_dev_col
        
        # Trabajamos con Series para mantener la alineación de fechas
        val_actual = df_corr_dev[col]
        val_prev = val_actual.shift(1)
        val_next = val_actual.shift(-1)
        
        # La corrección propuesta: promedio de vecinos (suavizado lineal)
        promedio_vecinos = (val_prev + val_next) / 2
        
        # Filtramos donde corregir
        mask_a_corregir = mask_corr_dev & val_actual.notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_corr_dev.index[mask_a_corregir]
            
            # --- EVALUACIÓN VS ORIGINAL (Evita el AssertionError) ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            
            if not idx_comun.empty:
                true_vals = df_original_eval.loc[idx_comun, col]
                imputed_vals = promedio_vecinos.loc[idx_comun]
                
                v_mask = true_vals.notna() & imputed_vals.notna()
                if v_mask.any():
                    all_true_corr_dev.extend(true_vals[v_mask].tolist())
                    all_imputed_corr_dev.extend(imputed_vals[v_mask].tolist())
            
            # --- APLICACIÓN DE LA CORRECCIÓN ---
            df_corr_dev.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_corregidos_por_columna[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y GRÁFICA ---
    if puntos_corregidos_por_columna:
        print("\nResumen de Correcciones:")
        for col, count in puntos_corregidos_por_columna.items():
            print(f"   - {col}: {count} valores corregidos.")
        
        if all_true_corr_dev:
            rmse_corr = np.sqrt(mean_squared_error(all_true_corr_dev, all_imputed_corr_dev))
            print(f"\n--- Evaluación de Calidad ---")
            print(f"RMSE Global: {rmse_corr:.3f} (sobre {len(all_true_corr_dev)} puntos)")
            
            plt.figure(figsize=(6, 6))
            plt.scatter(all_true_corr_dev, all_imputed_corr_dev, alpha=0.5, s=15, c='orange')
            mn, mx = min(all_true_corr_dev), max(all_true_corr_dev)
            plt.plot([mn, mx], [mn, mx], 'k--', lw=2, label='Ideal')
            plt.title("Desviación de Correlación: Real vs Corregido")
            plt.xlabel("Real"); plt.ylabel("Corregido"); plt.grid(True); plt.show()
    else:
        print("\nNo se realizaron correcciones efectivas.")

# Finalizar
df_corr_dev = df_corr_dev.reset_index()
datasets_listos[nombre_objetivo] = df_corr_dev

print("\n✅ FASE 5 COMPLETADA.")

## 12. Corrección — Desviación de Correlación

In [ ]:

# NOTA: En Fase 7 (datos reales sin inyección) no existe ground truth conocido.
# El RMSE/MAE aquí NO mide calidad de corrección — mide cuánto cambia el valor detectado.
# La evaluación cuantitativa válida está en Fases 1-6 (datos sintéticos con ground truth).
# Aquí la validación es cualitativa: inspección visual por experto del dominio.
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'ANOMALÍAS CONTEXTUALES' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Sincronización Crítica) ---
nombre_objetivo = 'combined_2024_03_06-2025_11_30_1min'
df_contextual = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para asegurar que df_contextual y df_original coincidan exactamente
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_contextual = preparar_y_redondear(df_contextual)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. LÓGICA DE CORRECCIÓN ---
factor_umbral_diferencia_ctx = FACTOR_UMBRAL_CORRECCION
nombre_clase_ctx = "Contextual"

# Máscara global según la predicción del modelo
mask_ctx_global = df_contextual['etiqueta_tipo_anomalia'] == nombre_clase_ctx

puntos_corregidos_por_columna = {}
all_true_ctx = []
all_imputed_ctx = []

if not mask_ctx_global.any():
    print(f"\nNo se detectaron anomalías de tipo '{nombre_clase_ctx}'.")
else:
    print(f"-> Filas marcadas como Contextuales: {mask_ctx_global.sum()}")
    
    for col in [c for c in columnas_objetivo if c in df_contextual.columns]:
        std_dev_col = df_original_eval[col].std()
        umbral = factor_umbral_diferencia_ctx * std_dev_col
        
        # Trabajamos con Series para mantener la alineación de fechas
        val_actual = df_contextual[col]
        val_prev = val_actual.shift(1)
        val_next = val_actual.shift(-1)
        
        # Corrección propuesta: promedio de vecinos
        promedio_vecinos = (val_prev + val_next) / 2
        
        # Filtramos donde corregir
        mask_a_corregir = mask_ctx_global & val_actual.notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_contextual.index[mask_a_corregir]
            
            # --- EVALUACIÓN VS ORIGINAL (Evita el AssertionError) ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            
            if not idx_comun.empty:
                true_vals = df_original_eval.loc[idx_comun, col]
                imputed_vals = promedio_vecinos.loc[idx_comun]
                
                v_mask = true_vals.notna() & imputed_vals.notna()
                if v_mask.any():
                    all_true_ctx.extend(true_vals[v_mask].tolist())
                    all_imputed_ctx.extend(imputed_vals[v_mask].tolist())
            
            # --- APLICACIÓN DE LA CORRECCIÓN ---
            df_contextual.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_corregidos_por_columna[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y EVALUACIÓN ---
    if puntos_corregidos_por_columna:
        print("\nResumen de Correcciones por Columna:")
        for col, count in puntos_corregidos_por_columna.items():
            print(f"   - {col}: {count} valores corregidos.")
        
        if all_true_ctx:
            rmse_ctx = np.sqrt(mean_squared_error(all_true_ctx, all_imputed_ctx))
            mae_ctx = mean_absolute_error(all_true_ctx, all_imputed_ctx)
            
            print(f"\n--- Evaluación de Calidad (Anomalía Contextual) ---")
            print(f"RMSE Global: {rmse_ctx:.3f} | MAE Global: {mae_ctx:.3f} (sobre {len(all_true_ctx)} puntos)")
            
            plt.figure(figsize=(6, 6))
            plt.scatter(all_true_ctx, all_imputed_ctx, alpha=0.5, s=15, c='darkblue')
            mn, mx = min(all_true_ctx), max(all_true_ctx)
            plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Ideal')
            plt.title("Anomalías Contextuales: Real vs Corregido")
            plt.xlabel("Valores Verdaderos (Original)")
            plt.ylabel("Valores Corregidos")
            plt.grid(True, linestyle='--', alpha=0.5); plt.legend(); plt.show()
    else:
        print("\nNo se realizaron correcciones efectivas en esta fase.")

# Devolvemos el DataFrame al diccionario
df_contextual = df_contextual.reset_index()
datasets_listos[nombre_objetivo] = df_contextual

print("\n✅ FASE 6 COMPLETADA. El dataset ha sido purificado de Anomalías Contextuales.")
print("🎉 EL MODELO 3 HA FINALIZADO TODAS SUS FASES EXITOSAMENTE 🎉")

## 13. Desglose final de anomalías detectadas

In [ ]:
print("=== DESGLOSE DE ANOMALÍAS DETECTADAS POR TIPO ===")

nombre_objetivo = "combined_2024_03_06-2025_11_30_1min"

if nombre_objetivo not in datasets_listos:
    print(f"No se encontró el dataset {nombre_objetivo} en datasets_listos.")
else:
    df_eval = datasets_listos[nombre_objetivo].copy()

    # --- 1. Resumen global de detección ---
    total_puntos = len(df_eval)
    total_anomalias = (df_eval["etiqueta_deteccion"] == "anomalia").sum()
    total_normales = total_puntos - total_anomalias
    print(f"
Total puntos evaluados : {total_puntos:,}")
    print(f"Normales               : {total_normales:,} ({total_normales/total_puntos*100:.1f}%)")
    print(f"Anomalías detectadas   : {total_anomalias:,} ({total_anomalias/total_puntos*100:.1f}%)")

    # --- 2. Desglose por tipo de anomalía ---
    if "etiqueta_tipo_anomalia" in df_eval.columns:
        df_anomalias = df_eval[df_eval["etiqueta_deteccion"] == "anomalia"]
        conteo_tipos = df_anomalias["etiqueta_tipo_anomalia"].value_counts()
        pct_tipos = (conteo_tipos / total_anomalias * 100).round(1)

        df_desglose = pd.DataFrame({
            "Tipo": conteo_tipos.index,
            "Cantidad": conteo_tipos.values,
            "% sobre anomalías": pct_tipos.values,
            "% sobre total": (conteo_tipos.values / total_puntos * 100).round(2)
        })

        print("
--- Desglose por tipo de anomalía ---")
        print(df_desglose.to_string(index=False))

        # --- 3. Gráfica de barras ---
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        colores = plt.cm.tab10.colors

        axes[0].barh(df_desglose["Tipo"], df_desglose["Cantidad"], color=colores[:len(df_desglose)])
        axes[0].set_xlabel("Cantidad de puntos")
        axes[0].set_title("Anomalías por tipo (conteo)")
        for i, v in enumerate(df_desglose["Cantidad"]):
            axes[0].text(v + total_anomalias*0.005, i, f"{v:,}", va="center", fontsize=9)

        wedges, texts, autotexts = axes[1].pie(
            df_desglose["Cantidad"],
            labels=df_desglose["Tipo"],
            autopct="%1.1f%%",
            colors=colores[:len(df_desglose)],
            startangle=140
        )
        axes[1].set_title("Distribución por tipo (%)")

        plt.suptitle(f"Desglose de {total_anomalias:,} anomalías detectadas — {nombre_objetivo}", fontsize=12, y=1.02)
        plt.tight_layout()
        os.makedirs("data/fase7", exist_ok=True)
        plt.savefig(f"data/fase7/desglose_anomalias_{nombre_objetivo}.png", bbox_inches="tight", dpi=150)
        plt.close()
        print(f"  Gráfica guardada en data/fase7/desglose_anomalias_{nombre_objetivo}.png")

        # --- 4. Desglose por tipo Y por columna de sensor ---
        columnas_sensores = [c for c in df_eval.columns
                             if c not in ["Fecha", "etiqueta_deteccion", "etiqueta_tipo_anomalia",
                                          "Hora", "DiaSemana", "Mes"]]

        print("
--- Anomalías 'Valores Fuera de Rango' por columna ---")
        mask_oor = (df_eval["etiqueta_tipo_anomalia"] == "Valores Fuera de Rango")
        if mask_oor.any():
            df_oor = df_eval[mask_oor]
            print(f"Total puntos Fuera de Rango: {mask_oor.sum():,}")
            for col in columnas_sensores:
                if col in df_oor.columns:
                    n_oor_col = df_oor[col].notna().sum()
                    if n_oor_col > 0:
                        print(f"  {col}: {n_oor_col:,} puntos fuera de rango")
        else:
            print("  No se encontraron puntos clasificados como 'Valores Fuera de Rango'.")
    else:
        print("
No existe la columna etiqueta_tipo_anomalia. Asegúrate de haber ejecutado el Modelo 2 (celda 128) antes.")

## 14. Resumen final

In [ ]:
os.makedirs(DATA_PLOTS, exist_ok=True)
print(f"Gráficas disponibles en: {DATA_PLOTS}")

nombre_objetivo = DATASET_NAME
if nombre_objetivo in datasets_listos:
    df_final = datasets_listos[nombre_objetivo]
    total = len(df_final)
    anomalias = (df_final['etiqueta_deteccion'] == 'anomalia').sum()
    print(f"\nResumen final:")
    print(f"  Total filas analizadas : {total:,}")
    print(f"  Anomalías detectadas   : {anomalias:,} ({anomalias/total*100:.2f}%)")
    if 'etiqueta_tipo_anomalia' in df_final.columns:
        print(f"\nDesglose por tipo:")
        print(df_final[df_final['etiqueta_deteccion']=='anomalia']['etiqueta_tipo_anomalia'].value_counts())
